# Qwen3-8B (như bài báo) trên GPU Kaggle/Colab

Chạy các thí nghiệm bằng Qwen3-8B (Ollama, bản Q8) trên GPU; graph đọc từ `outputs/hierargraph.pkl` trong repo (không dùng Neo4j). Notebook tự nhận ra đang ở Kaggle hay Colab.

Graph, án test, câu hỏi và cache embedding nằm sẵn trong repo; notebook chỉ cần clone, không cần tải dữ liệu lên.

## Kaggle (khuyên dùng: chạy nền, không cần mở trình duyệt)
1. Import notebook này vào Kaggle (Create → Import Notebook).
2. **Settings**: Accelerator = GPU T4 x2 (hoặc P100), Internet = On (cần xác minh số điện thoại tài khoản Kaggle).
3. Chạy thử từng ô một lần (Interactive) tới ô thử Qwen; ổn thì **Save Version → Save & Run All (Commit)**: chạy nền tối đa 12 giờ, có thể tắt máy.
4. Kết quả ở tab **Output** của version: thư mục `outputs/` (kết quả, log từng lượt) và `cache_llm.tar.gz`. Chạy tiếp (khi quá 12 giờ hoặc lỗi): **Add Input** → output của version trước, bỏ các lượt đã xong khỏi `RUNS`; notebook giải nén cache, các lời gọi đã xong không gọi lại.

## Colab
Runtime = T4 GPU; chạy từng ô (phải giữ tab mở). Cache để trên Drive.

Chọn lượt chạy ở biến `RUNS` (một version 12 giờ có thể không đủ cho cả 5 lượt). Chấm kết luận hỏi–đáp (`--no-grade` ở đây) làm sau ở máy bằng deepseek, để mọi lượt chạy cùng một người chấm.

In [ ]:
import glob, os, shutil, subprocess, sys, time, urllib.request
KAGGLE = os.path.exists('/kaggle')
!nvidia-smi --query-gpu=name,memory.total --format=csv

if KAGGLE:
    WORK = '/kaggle/working'
    OUT = WORK                       # outputs/ và cache_llm/ ở đây thành Output của version
    old_caches = glob.glob('/kaggle/input/**/cache_llm.tar.gz', recursive=True)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content'
    OUT = '/content/drive/MyDrive/KhoaLuan'
    old_caches = []
# Kaggle: cache ngoài Output (hàng nghìn file nhỏ), gom thành cache_llm.tar.gz sau mỗi lượt
CACHE = '/kaggle/tmp/cache_llm' if KAGGLE else f'{OUT}/cache_llm'
os.makedirs(f'{OUT}/outputs', exist_ok=True)
os.makedirs(CACHE, exist_ok=True)
for old in old_caches:               # Kaggle: tiếp tục từ output của version trước
    subprocess.check_call(['tar', 'xzf', old, '-C', CACHE, '--skip-old-files'])
    print(f'Đã giải nén cache từ {old}')
print('Kaggle' if KAGGLE else 'Colab', '| cache:', len(os.listdir(CACHE)), 'lời gọi')

In [ ]:
BRANCH = 'claude/legalgraphrag-framework-hv23z3'
REPO = f'{WORK}/Agent-Graph-Rag'
!rm -rf {REPO} && git clone -q -b {BRANCH} https://github.com/bighousevn/Agent-Graph-Rag.git {REPO}
os.chdir(REPO)
!pip install -q -r requirements.txt
!ls -la outputs/hierargraph.pkl data/processed/cases_vn_test.json data/raw/questions.xlsx

## Biến môi trường

In [ ]:
MODEL = 'qwen3:8b-q8_0'  # Qwen3-8B như bài báo; Q8 vừa một GPU 16 GB cùng ngữ cảnh 16k
os.environ.update({
    'LLM_PROVIDER': 'ollama',
    'LLM_API_KEY': 'ollama',          # Ollama không cần key; biến này chỉ để chọn đúng nhà cung cấp
    'LLM_BASE_URL': 'http://localhost:11434/v1',
    'LLM_MODEL': MODEL,
    'LLM_TIMEOUT': '600',
    'CACHE_DIR': CACHE,
})
print('Đã đặt biến môi trường cho', MODEL)

## Ollama + Qwen3-8B

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
env = {**os.environ, 'OLLAMA_CONTEXT_LENGTH': '16384', 'OLLAMA_KEEP_ALIVE': '-1'}
server = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open(f'{WORK}/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:11434'); break
    except Exception:
        time.sleep(1)
!ollama pull {MODEL} 2>&1 | tail -1

In [ ]:
# Thử một lời gọi: phải ra JSON, không có <think>
sys.path.insert(0, REPO)
from vn_legal_graph.config import AppConfig
from vn_legal_graph.llm import LLMClient
c = LLMClient(AppConfig.from_env_file('/nonexistent'))
c.config.cache_llm_calls = False
t = time.time()
print(c.generate('Trả về JSON {"toi_danh": [...]} cho hành vi: lén lấy xe máy của hàng xóm.'), f'({time.time() - t:.1f}s)')

## Chạy thí nghiệm

| # | lượt | ước lượng trên T4 |
|---|---|---|
| 1 | hỏi–đáp, Qwen không graph (187 câu) | ~15 phút |
| 2 | hỏi–đáp, pipeline graph (187 câu) | vài giờ |
| 3 | vụ án, Qwen không graph (55 án) | ~10 phút |
| 4 | vụ án, như bài gốc: Auditor lọc cứng | vài giờ |
| 5 | vụ án, bỏ Auditor + Adjudicator xem án tương tự | ~1 giờ (dùng lại cache của 4) |

Mỗi lượt xong thì kết quả được chép vào `outputs/` của Output (Kaggle) hoặc Drive (Colab).

In [ ]:
RUNS = [1, 2, 3, 4, 5]
COMMANDS = {
    1: 'python scripts/run_qa_pilot.py --auto-gold --n 1000 --without-graph --no-grade --tag full',
    2: 'python scripts/run_qa_pilot.py --auto-gold --n 1000 --rerank --no-grade --tag full',
    3: 'python scripts/run_case_pipeline.py --without-graph',
    4: 'python scripts/run_case_pipeline.py --auditor loc --tag loc',
    5: 'python scripts/run_case_pipeline.py --auditor bo-qua --adjudicator-context an --tag boqua_an',
}

def save():
    for f in glob.glob('outputs/*.json'):
        shutil.copy(f, f'{OUT}/outputs/')
    if KAGGLE:
        subprocess.check_call(['tar', 'czf', f'{OUT}/cache_llm.tar.gz', '-C', CACHE, '.'])

for r in RUNS:
    t = time.time()
    print(f'\n===== Lượt {r}: {COMMANDS[r]}', flush=True)
    log = f'{OUT}/outputs/luot{r}.log'
    code_ = subprocess.call(f'{COMMANDS[r]} > {log} 2>&1', shell=True)
    save()
    lines = open(log, encoding='utf-8').read().splitlines()
    tail = [l for l in lines if not l.startswith('  ')][-14:]
    print('\n'.join(tail))
    print(f'===== Lượt {r}: mã thoát {code_}, {(time.time() - t) / 60:.0f} phút', flush=True)

## Sau khi chạy xong (ở máy)

Chép `outputs/*.json` từ Output (Kaggle) hoặc Drive về `outputs/` của repo, rồi chấm kết luận hỏi–đáp bằng deepseek:

```
python scripts/run_qa_pilot.py --grade-only outputs/qa_llm_only_qwen3_8b-q8_0_full.json
python scripts/run_qa_pilot.py --grade-only outputs/qa_pilot_qwen3_8b-q8_0_full.json
```

Điểm vụ án (accuracy, F1) đã có trong file, không cần LLM chấm.